# ThoughtZero: Gemma 4 server on Colab

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AkulRanjan/pyschic-vla/blob/main/notebooks/colab_gemma.ipynb)

1. **Runtime → Change runtime type → a GPU** (T4 on the free tier; L4 or A100 with Colab Pro).
2. **Runtime → Run all.** The first run takes ~10–20 min (vLLM install + model download).
3. Choose a mode in the first code cell:
   - `MODE = "tunnel"` (default): serve Gemma here and expose it through a private,
     password-protected HTTPS tunnel. The last cell prints three lines; paste them into your
     local `.env`. Experiments then run **on your machine**, and your Jev key never leaves it.
   - `MODE = "in_colab"`: also run a script here (e.g. the real smoke test). Needs a Colab
     secret (🔑 icon on the left) named after your Jev route's key, e.g. `OPENROUTER_API_KEY`,
     with notebook access switched on.

Keep this tab open while you use the server: Colab stops idle sessions (~90 min) and every
session after at most 12 h (free tier). **Clear all outputs before sharing**: the tunnel
password is printed.

In [ ]:
# --- Parameters ---------------------------------------------------------------
MODE = "tunnel"  # "tunnel" | "in_colab"
REPO = "https://github.com/AkulRanjan/pyschic-vla.git"
BRANCH = "main"
JEV_TRANSPORT = "openrouter"  # in_colab only: direct | openrouter | zen | vercel | bocha
SCRIPT = "scripts/smoke_test.py --n-sims 4 --k 3"  # in_colab only

In [ ]:
# --- Pick weights for this GPU (docs/gpu_setup.md) -----------------------------
# Never import torch / transformers in this notebook's own process: the next cell's
# `pip install vllm` replaces torch on disk, and a torch already loaded in memory then
# clashes with the new files. Everything that needs them runs in a fresh subprocess.
import subprocess
import sys

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
    capture_output=True,
    text=True,
)
assert smi.returncode == 0, "No GPU: Runtime -> Change runtime type -> GPU"
name, mem_mib, cap = (x.strip() for x in smi.stdout.splitlines()[0].split(","))
major, mem_gb = int(cap.split(".")[0]), int(mem_mib) / 1024
TOKENIZER = "google/gemma-4-E4B-it"
if major >= 8 and mem_gb >= 22:  # L4 / A100: the bf16 weights (16 GB) fit
    CANDIDATES = [("google/gemma-4-E4B-it", ["--dtype", "bfloat16"])]
else:
    # Tested on a Colab T4 with vLLM 0.31 (2026-10-05): the 4-bit QAT checkpoint fails at
    # engine start-up, and vLLM no longer supports bitsandbytes. bf16 weights don't fit.
    raise SystemExit(
        f"{name} can't serve Gemma 4 E4B with vLLM: pick an L4 or A100 runtime (Colab Pro), "
        "or use a hosted generator instead (OpenRouter / Gemini API; see .env.example)."
    )
print(f"{name}: compute {major}.x, {mem_gb:.0f} GB -> will try {[m for m, _ in CANDIDATES]}")

In [ ]:
# --- Install vLLM (Gemma 4 needs >= 0.19.1; docs/verified_apis.md G5) --------------
!pip install -q -U "vllm>=0.19.1" bitsandbytes
!python -c "import importlib.metadata as m; print('vllm', m.version('vllm'))"

In [ ]:
# --- Start the server: try each candidate until one gives sane output -------------
import json
import secrets
import time
import urllib.request

API_KEY = secrets.token_hex(24)  # server password; hex, so it can never start with "-"

# Build the test prompt with Gemma's chat template, in a fresh process (see cell 2)
PROMPT = json.loads(
    subprocess.run(
        [
            sys.executable,
            "-c",
            f'''
import json
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("{TOKENIZER}")
print(json.dumps(tok.apply_chat_template(
    [{{"role": "user", "content": "What is 2+3? Put the final answer in \\\\boxed{{}}."}}],
    tokenize=False, add_generation_prompt=True, enable_thinking=False)))
''',
        ],
        capture_output=True,
        text=True,
        check=True,
    )
    .stdout.strip()
    .splitlines()[-1]
)
print("test prompt:", repr(PROMPT))


def call(path, body, timeout=300):
    req = urllib.request.Request(
        f"http://127.0.0.1:8000/v1/{path}",
        data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json", "Authorization": f"Bearer {API_KEY}"},
    )
    return json.load(urllib.request.urlopen(req, timeout=timeout))


def start(model, extra):
    log = open("vllm.log", "w")  # noqa: SIM115 (outlives this cell)
    cmd = [
        "vllm",
        "serve",
        model,
        *extra,
        "--max-model-len",
        "8192",
        "--enable-prefix-caching",
        "--max-logprobs",
        "20",
        f"--api-key={API_KEY}",
        "--host",
        "127.0.0.1",
        "--port",
        "8000",
    ]
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(360):  # up to 30 min: the first download is slow
        if proc.poll() is not None:
            return proc, False
        try:
            urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2)
            return proc, True
        except Exception:
            time.sleep(5)
    return proc, False


server = MODEL = None
for model, extra in CANDIDATES:
    print(f"starting {model} {' '.join(extra)} ...")
    proc, up = start(model, extra)
    if up:
        body = {
            "model": model,
            "prompt": PROMPT,
            "max_tokens": 64,
            "temperature": 0,
            "logprobs": 20,
        }
        out = call("completions", body)["choices"][0]
        text = out["text"]
        printable = sum(c.isprintable() or c.isspace() for c in text) / max(1, len(text))
        sane = "5" in text and printable > 0.95
        top_k = len(((out.get("logprobs") or {}).get("top_logprobs") or [{}])[0] or {})
        print(f"  output: {text!r}\n  sane={sane}  logprobs top-k={top_k}")
        if sane:
            server, MODEL = proc, model
            break
    with open("vllm.log") as f:
        print("  failed; last log lines:\n" + "".join(f.readlines()[-15:]))
    proc.terminate()
    proc.wait()
assert server is not None, "no candidate served sane output; see vllm.log"

t = time.time()
body = {
    "model": MODEL,
    "prompt": PROMPT,
    "max_tokens": 128,
    "temperature": 0.9,
    "top_p": 0.95,
    "n": 4,
}
r = call("completions", body)
print(
    f"throughput, n=4 short steps: {r['usage']['completion_tokens'] / (time.time() - t):.0f} tok/s"
)

In [ ]:
# --- MODE = "tunnel": expose the server, print the .env lines ----------------------
import re

if MODE == "tunnel":
    url = (
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    )
    subprocess.run(["wget", "-q", "-nc", url, "-O", "cloudflared"], check=False)
    subprocess.run(["chmod", "+x", "cloudflared"], check=True)
    tunnel = subprocess.Popen(
        ["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    public = None
    for line in tunnel.stdout:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            public = m.group(0)
            break
    print("Paste these into your local .env (and keep this tab open):\n")
    print(f"GEMMA_BASE_URL={public}/v1")
    print(f"GEMMA_MODEL={MODEL}")
    print(f"GEMMA_TOKENIZER={TOKENIZER}")
    print(f"GEMMA_API_KEY={API_KEY}")

In [ ]:
# --- MODE = "in_colab": run a script here, with the Jev key from Colab secrets --------
if MODE == "in_colab":
    from google.colab import userdata

    KEYS = {
        "direct": "TYPESAFE_API_KEY",
        "openrouter": "OPENROUTER_API_KEY",
        "zen": "OPENCODE_ZEN_API_KEY",
        "vercel": "AI_GATEWAY_API_KEY",
        "bocha": "BOCHA_API_KEY",
    }
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO, "repo"], check=True)
    # --no-deps keeps vLLM's torch/transformers; add only what vLLM doesn't bring
    subprocess.run(["pip", "install", "-q", "--no-deps", "-e", "repo"], check=True)
    subprocess.run(
        ["pip", "install", "-q", "diskcache", "math-verify", "python-dotenv", "tenacity"],
        check=True,
    )
    key_name = KEYS[JEV_TRANSPORT]
    with open("repo/.env", "w") as f:
        f.write(f"GEMMA_BASE_URL=http://127.0.0.1:8000/v1\nGEMMA_MODEL={MODEL}\n")
        f.write(f"GEMMA_TOKENIZER={TOKENIZER}\nGEMMA_API_KEY={API_KEY}\n")
        f.write(f"JEV_TRANSPORT={JEV_TRANSPORT}\n{key_name}={userdata.get(key_name)}\n")
    subprocess.run(f"cd repo && python {SCRIPT}", shell=True, check=False)